# 04 - Ranquear candidatos SWOT por proximidade

Este notebook melhora a seleção de candidatos SWOT usando a geometria disponível nos metadados Earthdata/PO.DAAC, antes de qualquer download PIXC pesado.

## Por que o RiverSP anterior foi rejeitado?

O candidato RiverSP rank 1 escolhido por data recente e envelope de busca foi baixado e aberto, mas a feição RiverSP mais próxima ficou a aproximadamente 162 km dos 13 exutórios. Isso mostra que cobertura por envelope/catalogação não garante suporte observacional real no ponto.

## Cobertura por envelope vs proximidade real

A busca por `bounding_box` retorna grânulos que intersectam uma área ampla. A proximidade real exige comparar os pontos dos exutórios com a geometria espacial do grânulo, preferindo polígonos de cobertura quando disponíveis e usando bounding boxes apenas como aproximação.

In [ ]:
from __future__ import annotations

import logging
import re
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Polygon, box
from shapely.ops import unary_union


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
EXUTORIOS_CSV = PROJECT_ROOT / 'dados' / 'exutorios.csv'
CANDIDATOS_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'candidatos_passagem_teste_swot.csv'
VALIDACAO_RIVERSP_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'validacao_riversp_exutorios.csv'
OUTPUT_RANKING = PROJECT_ROOT / 'outputs' / 'tabelas' / 'ranking_candidatos_swot_proximidade.csv'
OUTPUT_BY_POINT = PROJECT_ROOT / 'outputs' / 'tabelas' / 'ranking_candidatos_por_exutorio.csv'
OUTPUT_FIGURE = PROJECT_ROOT / 'outputs' / 'figuras' / '04_ranking_candidatos_swot_proximidade.png'
LOG_FILE = PROJECT_ROOT / 'outputs' / 'logs' / '04_ranquear_candidatos_por_proximidade.log'

for path in [OUTPUT_RANKING.parent, OUTPUT_BY_POINT.parent, OUTPUT_FIGURE.parent, LOG_FILE.parent]:
    path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(filename=LOG_FILE, filemode='w', level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('OK raiz do projeto:', PROJECT_ROOT)
print('OK log:', LOG_FILE)


## Entradas

São lidos os 13 exutórios, a tabela de candidatos do notebook 02 quando existir, e a validação RiverSP do notebook 03 quando existir. A validação anterior é usada para destacar o candidato já rejeitado.

In [ ]:
if not EXUTORIOS_CSV.exists():
    raise FileNotFoundError(f'FALHA: exutorios.csv nao encontrado: {EXUTORIOS_CSV}')

exutorios = pd.read_csv(EXUTORIOS_CSV)
expected_columns = ['id', 'latitude', 'longitude']
if list(exutorios.columns) != expected_columns:
    raise ValueError(f'FALHA: colunas esperadas {expected_columns}, colunas encontradas {list(exutorios.columns)}')
if len(exutorios) != 13:
    raise ValueError(f'FALHA: esperados 13 exutorios, encontrados {len(exutorios)}')
exutorios['latitude'] = pd.to_numeric(exutorios['latitude'], errors='raise')
exutorios['longitude'] = pd.to_numeric(exutorios['longitude'], errors='raise')

candidatos_anteriores = pd.read_csv(CANDIDATOS_CSV) if CANDIDATOS_CSV.exists() else pd.DataFrame()
validacao_riversp = pd.read_csv(VALIDACAO_RIVERSP_CSV) if VALIDACAO_RIVERSP_CSV.exists() else pd.DataFrame()
rejected_granule_ids = set()
if not candidatos_anteriores.empty and not validacao_riversp.empty:
    if 'suporte_riversp' in validacao_riversp.columns and (validacao_riversp['suporte_riversp'].astype(str).str.lower() == 'nao').all():
        rejected_granule_ids.update(candidatos_anteriores.loc[candidatos_anteriores['produto'].str.upper() == 'RIVERSP', 'granule_id'].astype(str))

points_gdf = gpd.GeoDataFrame(
    exutorios.copy(),
    geometry=gpd.points_from_xy(exutorios['longitude'], exutorios['latitude']),
    crs='EPSG:4326',
)
metric_crs = points_gdf.estimate_utm_crs() or 'EPSG:32723'
points_m = points_gdf.to_crs(metric_crs)

BUFFER_METERS = 5000
buffered_union_m = points_m.geometry.buffer(BUFFER_METERS).union_all()
bbox_wgs84 = gpd.GeoSeries([buffered_union_m.envelope], crs=metric_crs).to_crs('EPSG:4326').iloc[0]
bbox = tuple(bbox_wgs84.bounds)

logging.info('Exutorios: %s', len(points_gdf))
logging.info('CRS metrico: %s', metric_crs)
logging.info('Bounding box WGS84: %s', bbox)
logging.info('Granulos RiverSP rejeitados: %s', sorted(rejected_granule_ids))
print('OK exutorios:', len(points_gdf))
print('OK CRS metrico:', metric_crs)
print('OK bounding_box:', bbox)
print('RiverSP rejeitado anteriormente:', sorted(rejected_granule_ids) or 'nenhum')
display(points_gdf)


## Critérios de ranqueamento

O ranking prioriza: mais exutórios dentro da geometria do grânulo; menor distância mínima e mediana; produtos leves em caso de empate; data mais recente como critério secundário. Quando só houver bounding box, o resultado é marcado como aproximação.

In [ ]:
try:
    import earthaccess
except Exception as exc:
    logging.exception('Falha ao importar earthaccess')
    raise RuntimeError('FALHA: earthaccess nao esta instalado. Rode pip install -r requirements.txt.') from exc

START_DATE = '2023-01-01'
END_DATE = datetime.now(timezone.utc).date().isoformat()
PRODUCTS = {
    'SWOT_L2_HR_RIVERSP_D': {'pattern': '*Reach*', 'produto': 'RIVERSP', 'peso_produto': 1},
    'SWOT_L2_HR_LAKESP_D': {'pattern': '*Prior*', 'produto': 'LAKESP', 'peso_produto': 2},
    'SWOT_L2_HR_PIXC_D': {'pattern': '*', 'produto': 'PIXC', 'peso_produto': 3},
}

def granule_umm(granule) -> dict:
    if hasattr(granule, 'umm'):
        return granule.umm
    if isinstance(granule, dict):
        return granule.get('umm', granule)
    try:
        return granule['umm']
    except Exception:
        return {}

def granule_native_id(granule) -> str:
    umm = granule_umm(granule)
    return umm.get('GranuleUR') or umm.get('ProducerGranuleId') or str(granule)

def parse_cycle_pass_tile(name: str) -> dict:
    patterns = [
        r'SWOT_L2_HR_PIXC_(\d{3})_(\d{3})_([0-9]{3}[LR])_',
        r'SWOT_L2_HR_RiverSP_Reach_(\d{3})_(\d{3})_([A-Z]{2})_',
        r'SWOT_L2_HR_LakeSP_Prior_(\d{3})_(\d{3})_([A-Z]{2})_',
    ]
    for pattern in patterns:
        match = re.search(pattern, name)
        if match:
            return {'cycle': match.group(1), 'pass': match.group(2), 'tile': match.group(3)}
    return {'cycle': None, 'pass': None, 'tile': None}

def get_time_bounds(umm: dict, name: str):
    range_time = ((umm.get('TemporalExtent') or {}).get('RangeDateTime') or {})
    start = pd.to_datetime(range_time.get('BeginningDateTime'), errors='coerce', utc=True)
    end = pd.to_datetime(range_time.get('EndingDateTime'), errors='coerce', utc=True)
    if pd.isna(start):
        matches = re.findall(r'(20\d{6}T\d{6})', name)
        if matches:
            start = pd.to_datetime(matches[0], format='%Y%m%dT%H%M%S', errors='coerce', utc=True)
        if len(matches) >= 2:
            end = pd.to_datetime(matches[1], format='%Y%m%dT%H%M%S', errors='coerce', utc=True)
    return start, end

def size_mb(umm: dict):
    info = (umm.get('DataGranule') or {}).get('ArchiveAndDistributionInformation', [])
    sizes = []
    for item in info or []:
        try:
            size = float(item.get('Size'))
        except Exception:
            continue
        unit = str(item.get('SizeUnit', 'MB')).upper()
        if unit.startswith('KB'):
            size /= 1024
        elif unit.startswith('GB'):
            size *= 1024
        elif unit in {'B', 'BYTE', 'BYTES'}:
            size /= 1024 * 1024
        sizes.append(size)
    return max(sizes) if sizes else None

def is_auxiliary_url(url: str) -> bool:
    return url.lower().endswith(('.met.json', '.iso.xml', '.log', '.png', '.jpg', '.jpeg', '.md5', '.sha256', '.xml', '.json', '.txt'))

def expected_suffix(produto: str):
    return ('.nc',) if produto == 'PIXC' else ('.zip',)

def download_url(umm: dict, produto: str) -> str:
    urls = [item.get('URL') for item in (umm.get('RelatedUrls') or []) if item.get('URL')]
    scientific = [u for u in urls if u.lower().endswith(expected_suffix(produto)) and not is_auxiliary_url(u)]
    if scientific:
        return scientific[0]
    non_aux = [u for u in urls if not is_auxiliary_url(u)]
    return non_aux[0] if non_aux else (urls[0] if urls else '')

def polygon_from_points(points):
    coords = []
    for pt in points or []:
        lon = pt.get('Longitude')
        lat = pt.get('Latitude')
        if lon is None or lat is None:
            continue
        coords.append((float(lon), float(lat)))
    if len(coords) < 3:
        return None
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    poly = Polygon(coords)
    return poly if poly.is_valid and not poly.is_empty else poly.buffer(0)

def geometry_from_umm(umm: dict):
    hsd = ((umm.get('SpatialExtent') or {}).get('HorizontalSpatialDomain') or {})
    geom = hsd.get('Geometry') or {}
    polygons = []
    for gpoly in geom.get('GPolygons', []) or []:
        boundary = gpoly.get('Boundary') or {}
        poly = polygon_from_points(boundary.get('Points') or [])
        if poly is not None and not poly.is_empty:
            polygons.append(poly)
    if polygons:
        return unary_union(polygons), 'bounding_polygon'
    rectangles = []
    for rect in geom.get('BoundingRectangles', []) or []:
        try:
            rectangles.append(box(float(rect['WestBoundingCoordinate']), float(rect['SouthBoundingCoordinate']), float(rect['EastBoundingCoordinate']), float(rect['NorthBoundingCoordinate'])))
        except Exception:
            continue
    if rectangles:
        return unary_union(rectangles), 'bounding_box'
    return None, 'sem_geometria'


In [ ]:
records = []
for short_name, cfg in PRODUCTS.items():
    logging.info('Consultando %s', short_name)
    try:
        results = earthaccess.search_data(short_name=short_name, temporal=(START_DATE, END_DATE), bounding_box=bbox, granule_name=cfg['pattern'], count=-1)
    except Exception as exc:
        logging.exception('Falha na consulta %s', short_name)
        raise RuntimeError(f'FALHA na consulta Earthdata/CMR para {short_name}.') from exc
    print(f'OK consulta {short_name}: {len(results)} granulos')
    for granule in results:
        umm = granule_umm(granule)
        granule_id = granule_native_id(granule)
        geom, geom_type = geometry_from_umm(umm)
        start, end = get_time_bounds(umm, granule_id)
        parsed = parse_cycle_pass_tile(granule_id)
        records.append({
            'produto': cfg['produto'], 'short_name': short_name, 'cycle': parsed['cycle'], 'pass': parsed['pass'], 'tile': parsed['tile'],
            'data_inicio': start, 'data_fim': end, 'tamanho_mb': size_mb(umm), 'granule_id': granule_id,
            'download_url': download_url(umm, cfg['produto']), 'peso_produto': cfg['peso_produto'],
            'tipo_geometria_usada': geom_type, 'geometry': geom,
            'foi_rejeitado_riversp': granule_id in rejected_granule_ids,
        })

candidates = gpd.GeoDataFrame(records, geometry='geometry', crs='EPSG:4326')
candidates = candidates[~candidates.geometry.isna()].copy()
if candidates.empty:
    raise RuntimeError('FALHA: nenhum candidato com geometria espacial foi retornado pelo CMR.')

candidates['data_inicio'] = pd.to_datetime(candidates['data_inicio'], errors='coerce', utc=True)
candidates['data_fim'] = pd.to_datetime(candidates['data_fim'], errors='coerce', utc=True)
candidates['tamanho_mb'] = pd.to_numeric(candidates['tamanho_mb'], errors='coerce')
candidates = candidates.drop_duplicates(subset=['produto', 'granule_id']).reset_index(drop=True)
print('OK candidatos com geometria:', len(candidates))
print(candidates['tipo_geometria_usada'].value_counts())
display(candidates[['produto','cycle','pass','tile','data_inicio','tamanho_mb','tipo_geometria_usada','granule_id']].head())


In [ ]:
candidates_m = candidates.to_crs(metric_crs)
point_rows = []
ranking_rows = []
for idx, cand in candidates_m.iterrows():
    distances = points_m.geometry.distance(cand.geometry)
    inside = points_m.geometry.within(cand.geometry) | points_m.geometry.intersects(cand.geometry)
    n_inside = int(inside.sum())
    dist_min = float(distances.min())
    dist_med = float(distances.median())
    original = candidates.loc[idx]
    if original['foi_rejeitado_riversp']:
        motivo = 'RiverSP ja testado: rejeitado por distancia real alta no notebook 03.'
    elif n_inside > 0:
        motivo = f'{n_inside} exutorios dentro/intersectando a geometria do granulo.'
    else:
        motivo = 'Nenhum exutorio dentro; ranking por menor distancia ate a geometria do granulo.'
    ranking_rows.append({
        'produto': original['produto'], 'cycle': original['cycle'], 'pass': original['pass'], 'tile': original['tile'],
        'data_inicio': original['data_inicio'], 'data_fim': original['data_fim'], 'tamanho_mb': original['tamanho_mb'],
        'n_exutorios_dentro': n_inside, 'distancia_min_m': round(dist_min, 2), 'distancia_mediana_m': round(dist_med, 2),
        'tipo_geometria_usada': original['tipo_geometria_usada'], 'granule_id': original['granule_id'], 'download_url': original['download_url'],
        'motivo_rank': motivo, 'peso_produto': original['peso_produto'], 'foi_rejeitado_riversp': bool(original['foi_rejeitado_riversp'])
    })
    for _, point in points_m.iterrows():
        d = float(point.geometry.distance(cand.geometry))
        point_rows.append({
            'id': point['id'], 'produto': original['produto'], 'cycle': original['cycle'], 'pass': original['pass'], 'tile': original['tile'],
            'distancia_m': round(d, 2), 'dentro_geometria': bool(point.geometry.within(cand.geometry) or point.geometry.intersects(cand.geometry)),
            'granule_id': original['granule_id']
        })

ranking = pd.DataFrame(ranking_rows)
ranking = ranking.sort_values(
    by=['foi_rejeitado_riversp','n_exutorios_dentro','distancia_min_m','distancia_mediana_m','peso_produto','data_inicio'],
    ascending=[True, False, True, True, True, False],
    na_position='last'
).reset_index(drop=True)
ranking.insert(0, 'rank', range(1, len(ranking)+1))

ranking_out = ranking[['rank','produto','cycle','pass','tile','data_inicio','data_fim','tamanho_mb','n_exutorios_dentro','distancia_min_m','distancia_mediana_m','tipo_geometria_usada','granule_id','download_url','motivo_rank']].copy()
for col in ['data_inicio','data_fim']:
    ranking_out[col] = pd.to_datetime(ranking_out[col], errors='coerce', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
ranking_out.to_csv(OUTPUT_RANKING, index=False, encoding='utf-8')

rank_lookup = ranking.set_index('granule_id')['rank'].to_dict()
by_point = pd.DataFrame(point_rows)
by_point['rank_global'] = by_point['granule_id'].map(rank_lookup)
by_point = by_point[by_point['rank_global'].le(10)].sort_values(['rank_global','id'])
by_point = by_point[['rank_global','id','produto','cycle','pass','tile','distancia_m','dentro_geometria','granule_id']]
by_point.to_csv(OUTPUT_BY_POINT, index=False, encoding='utf-8')

logging.info('Ranking salvo: %s', OUTPUT_RANKING)
logging.info('Ranking por exutorio salvo: %s', OUTPUT_BY_POINT)
print('OK ranking salvo:', OUTPUT_RANKING)
print('OK ranking por exutorio salvo:', OUTPUT_BY_POINT)
display(ranking_out.head(10))
display(by_point.head(30))


## Limitações de bounding box

Quando o CMR só fornece bounding box, a geometria pode ser bem mais ampla que a faixa útil do produto. Nesses casos, `n_exutorios_dentro` pode superestimar a observabilidade. Polígonos de cobertura são preferíveis; bounding boxes servem apenas como filtro preliminar.

In [ ]:
top = ranking.head(6).copy()
plot_ids = set(top['granule_id']).union(rejected_granule_ids)
plot_gdf = candidates[candidates['granule_id'].isin(plot_ids)].copy()
plot_gdf['rank'] = plot_gdf['granule_id'].map(ranking.set_index('granule_id')['rank'])
plot_gdf['label'] = plot_gdf['rank'].astype(str) + ' - ' + plot_gdf['produto'].astype(str)
colors = {'RIVERSP':'#1b9e77', 'LAKESP':'#7570b3', 'PIXC':'#d95f02'}
fig, ax = plt.subplots(figsize=(10, 9))
for produto, subset in plot_gdf.groupby('produto'):
    subset.boundary.plot(ax=ax, color=colors.get(produto, 'black'), linewidth=1.5, label=produto)
rejected = plot_gdf[plot_gdf['granule_id'].isin(rejected_granule_ids)]
if not rejected.empty:
    rejected.boundary.plot(ax=ax, color='red', linewidth=3.0, linestyle='--', label='RiverSP rejeitado')
points_gdf.plot(ax=ax, color='black', markersize=40, label='Exutorios')
for _, row in points_gdf.iterrows():
    ax.annotate(row['id'], (row.geometry.x, row.geometry.y), xytext=(3,3), textcoords='offset points', fontsize=8)
minx, miny, maxx, maxy = points_gdf.total_bounds
pad_x = max((maxx-minx)*8, 0.08)
pad_y = max((maxy-miny)*8, 0.08)
ax.set_xlim(minx-pad_x, maxx+pad_x)
ax.set_ylim(miny-pad_y, maxy+pad_y)
ax.set_title('Ranking SWOT por proximidade - melhores candidatos')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE, dpi=180)
logging.info('Figura salva: %s', OUTPUT_FIGURE)
print('OK figura salva:', OUTPUT_FIGURE)
plt.show()


## Recomendação do próximo candidato

Use `ranking_candidatos_swot_proximidade.csv` para escolher o próximo teste. Preferir um candidato RiverSP ou LakeSP com maior `n_exutorios_dentro` e menor distância. PIXC deve continuar como última opção por ser pesado, salvo se for necessário validar pixels diretamente.